In [16]:
# Add target col

import pandas as pd
import nba_api_module as nbpim

df_input = pd.read_csv("data/ml_ready_2024_25.csv")

# Drop star injury
df_input.drop(columns=['home_missing_starters', 'away_missing_starters'], inplace=True)


gid = f"00{df_input.loc[0, "game_id"]}"

season_log = nbpim.get_season_game_log("2024-25")
season_log_home = season_log[season_log['MATCHUP'].str.contains(' vs. ')]
season_log_home['is_home_win'] = season_log_home['WL'].apply(lambda x: 1 if x == 'W' else 0)
season_log_home.rename(columns={'GAME_ID': 'game_id'}, inplace=True)
season_log_home.game_id = season_log_home.game_id.astype(int)

df_input = pd.merge(df_input, season_log_home[['game_id', 'is_home_win']],
                    on="game_id", how="left")

print(df_input.sample(5))
print(df_input.columns)

      game_id  home_ORtg  away_ORtg  home_DRtg  away_DRtg  home_NET_rtg  \
21   22400022  87.313721  84.073894  92.286800  88.441171     -4.973079   
87   22400088  84.073894  91.772910  88.441171  92.498014     -4.367277   
387  22400388  87.882580  85.322353  92.608383  82.225978     -4.725802   
426  22400427  89.654286  91.772910  91.511610  92.498014     -1.857324   
128  22400129  93.665306  85.322353  91.872945  82.225978      1.792361   

     away_NET_rtg   home_PACE   away_PACE  home_TS%  ...  home_rest_days  \
21      -4.367277  120.277222  126.376923  0.556417  ...               5   
87      -0.725104  126.376923  119.696604  0.540116  ...               5   
387      3.096376  125.808364  132.119273  0.560040  ...               5   
426     -0.725104  130.197091  119.696604  0.569622  ...               5   
128      3.096376  123.540179  132.119273  0.587135  ...               4   

     home_recent_form10  home_recent_form5  home_recent_form3  \
21                  0.6    

C:\Users\Adam\AppData\Local\Temp\ipykernel_11528\1038124081.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  season_log_home['is_home_win'] = season_log_home['WL'].apply(lambda x: 1 if x == 'W' else 0)
C:\Users\Adam\AppData\Local\Temp\ipykernel_11528\1038124081.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  season_log_home.rename(columns={'GAME_ID': 'game_id'}, inplace=True)
C:\Users\Adam\AppData\Local\Temp\ipykernel_11528\1038124081.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_ind

In [18]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, log_loss
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier

# -------------------------------------------------------------------------
# 1) ADATOK ELŐKÉSZÍTÉSE
# -------------------------------------------------------------------------

# Feltételezzük, hogy df_input a dataframe-ed
df = df_input.copy()
df.dropna(subset=["is_home_win"], inplace=True)

# Célváltozó
y = df["is_home_win"]

# Nem kell információ: game_id
X = df.drop(columns=["is_home_win", "game_id"])

# Időrendi split (550 sor -> 80/20)
split_idx = int(len(df) * 0.8)

X_train = X.iloc[:split_idx]
X_test  = X.iloc[split_idx:]
y_train = y.iloc[:split_idx]
y_test  = y.iloc[split_idx:]

# Logistic Regression-hez szükséges standardizálás
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# -------------------------------------------------------------------------
# 2) MODELL LISTA
# -------------------------------------------------------------------------

models = {
    "Logistic Regression": LogisticRegression(max_iter=500),
    "Random Forest": RandomForestClassifier(n_estimators=300, max_depth=8),
    "Gradient Boosting": GradientBoostingClassifier(),
    "XGBoost": XGBClassifier(
        n_estimators=400,
        learning_rate=0.03,
        max_depth=6,
        subsample=0.9,
        colsample_bytree=0.9,
        eval_metric="logloss",
        use_label_encoder=False
    )
}

# -------------------------------------------------------------------------
# 3) MODELLEK KIÉRTÉKELÉSE
# -------------------------------------------------------------------------

results = {}

for name, model in models.items():

    # Logistic Regression -> scale-elt adat
    if name == "Logistic Regression":
        model.fit(X_train_scaled, y_train)
        pred_proba = model.predict_proba(X_test_scaled)[:, 1]
        preds = (pred_proba > 0.5).astype(int)
    else:
        model.fit(X_train, y_train)
        pred_proba = model.predict_proba(X_test)[:, 1]
        preds = (pred_proba > 0.5).astype(int)

    acc = accuracy_score(y_test, preds)
    auc = roc_auc_score(y_test, pred_proba)
    ll  = log_loss(y_test, pred_proba)

    results[name] = {
        "Accuracy": acc,
        "AUC": auc,
        "LogLoss": ll
    }

# -------------------------------------------------------------------------
# 4) EREDMÉNYEK KIÍRÁSA
# -------------------------------------------------------------------------

print("\n=== MODELL EREDMÉNYEK ===\n")
for name, metrics in results.items():
    print(f"{name}:")
    print(f"  Accuracy  : {metrics['Accuracy']:.4f}")
    print(f"  AUC       : {metrics['AUC']:.4f}")
    print(f"  LogLoss   : {metrics['LogLoss']:.4f}")
    print("-----------------------------------------")

# -------------------------------------------------------------------------
# 5) LEGJOBB MODELL KIVÁLASZTÁSA AUC ALAPJÁN
# -------------------------------------------------------------------------

best_model_name = max(results, key=lambda k: results[k]["AUC"])
print(f"\nLegjobb modell: {best_model_name}")


c:\Users\Adam\anaconda3\Lib\site-packages\xgboost\training.py:183: UserWarning: [09:49:26] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



=== MODELL EREDMÉNYEK ===

Logistic Regression:
  Accuracy  : 0.6273
  AUC       : 0.6741
  LogLoss   : 0.6551
-----------------------------------------
Random Forest:
  Accuracy  : 0.6545
  AUC       : 0.7006
  LogLoss   : 0.6366
-----------------------------------------
Gradient Boosting:
  Accuracy  : 0.6636
  AUC       : 0.6608
  LogLoss   : 0.7118
-----------------------------------------
XGBoost:
  Accuracy  : 0.6273
  AUC       : 0.6691
  LogLoss   : 0.8150
-----------------------------------------

Legjobb modell: Random Forest
